# Auto-encodeur Variationnel (VAE) pour la Generation de Chiffres MNIST

## Contexte et Objectifs

Ce notebook presente une implementation detaillee d'un Auto-encodeur Variationnel (VAE), un modele generatif puissant, pour apprendre la distribution des chiffres manuscrits du celebre jeu de donnees MNIST. Contrairement a un auto-encodeur classique qui apprend une representation compresse, un VAE apprend une distribution de probabilite dans l'espace latent, ce qui lui permet de generer de nouvelles donnees plausibles.

### Concepts Cles de ce Notebook :

1.  **Architecture Encodeur-Decodeur :** Nous construisons un VAE avec PyTorch, compose de deux parties :
    *   **L'Encodeur :** Il prend une image en entree et produit les parametres (moyenne et log-variance) d'une distribution gaussienne dans l'espace latent.
    *   **Le Decodeur :** Il prend un point echantillonne de l'espace latent et genere une nouvelle image.
2.  **L'Astuce de Reparametrisation :** C'est une technique cruciale qui permet a la retropropagation du gradient de fonctionner a travers l'etape d'echantillonnage, rendant l'entrainement du VAE possible.
3.  **Fonction de Perte du VAE :** La perte combine deux termes :
    *   **Perte de Reconstruction :** Mesure a quel point l'image reconstruite est fidele a l'originale.
    *   **Divergence de Kullback-Leibler (KL) :** Agit comme un regularisateur, forcant la distribution latente a se rapprocher d'une distribution predefinie (typiquement une gaussienne standard).
4.  **Generation et Visualisation :** Nous montrons comment :
    *   Reconstruire des images de test pour evaluer la qualite du modele.
    *   Generer de nouveaux chiffres en echantillonnant des points aleatoires de l'espace latent.
    *   Visualiser l'espace latent en 2D pour comprendre comment le modele organise les differents chiffres.

_Derniere mise a jour : 2026-02-16_

In [1]:
# --- 1. Installation des Dependances ---
%pip install -q torch torchvision matplotlib
print("Dependances installees.")

SyntaxError: invalid syntax (2051176013.py, line 1)

In [2]:
# --- 2. Imports et Configuration ---
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
import numpy as np
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SyntaxError: invalid syntax (164059663.py, line 1)

## 3. Preparation des Donnees MNIST

Nous chargeons le jeu de donnees MNIST et creons des `DataLoader` pour l'entrainement et le test. Les images sont normalisees pour avoir des valeurs de pixels entre 0 et 1.

In [3]:
# --- Hyperparametres ---
batch_size = 128
latent_dim = 2 # Dimension de l'espace latent (2D pour la visualisation)
epochs = 10
learning_rate = 1e-3

# --- Transformations et Chargement des Donnees ---
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

logger.info(f"{len(train_dataset)} images pour l'entrainement, {len(test_dataset)} pour le test.")

SyntaxError: invalid syntax (823652641.py, line 1)

## 4. Definition du Modele VAE

Le VAE est compose d'un encodeur et d'un decodeur, tous deux implementes comme des reseaux de neurones simples. La fonction `reparameterize` implemente l'astuce de reparametrisation.

In [4]:
class VAE(nn.Module):
    def __init__(self):
        super(VAE, self).__init__()

        # Encodeur
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(256, latent_dim) # Couche pour la moyenne (mu)
        self.fc_logvar = nn.Linear(256, latent_dim) # Couche pour la log-variance (log_var)

        # Decodeur
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 28 * 28),
            nn.Sigmoid() # Pour ramener les pixels entre 0 et 1
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x.view(-1, 28 * 28))
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

# Initialiser le modele et le deplacer sur le bon device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

SyntaxError: invalid syntax (3688422110.py, line 1)

## 5. Fonction de Perte et Boucle d'Entrainement

In [5]:
def loss_function(recon_x, x, mu, log_var):
    # Perte de reconstruction (Binary Cross-Entropy)
    BCE = nn.functional.binary_cross_entropy(recon_x, x.view(-1, 28 * 28), reduction='sum')
    # Divergence KL
    KLD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return BCE + KLD

def train(epoch):
    model.train()
    train_loss = 0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        recon_batch, mu, log_var = model(data)
        loss = loss_function(recon_batch, data, mu, log_var)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    
    avg_loss = train_loss / len(train_loader.dataset)
    logger.info(f'====> Epoch: {epoch} | Perte moyenne: {avg_loss:.4f}')

# Lancer l'entrainement
logger.info("Debut de l'entrainement du VAE...")
for epoch in range(1, epochs + 1):
    train(epoch)

SyntaxError: invalid syntax (416476538.py, line 1)

## 6. Resultats et Visualisations

Apres l'entrainement, nous pouvons utiliser le VAE pour reconstruire des images, en generer de nouvelles, et visualiser l'organisation de l'espace latent.

In [ ]:
def visualize_reconstructions(model, data_loader, device):
    model.eval()
    with torch.no_grad():
        data, _ = next(iter(data_loader))
        data = data.to(device)
        recon, _, _ = model(data)
        
        # Comparaison des images originales et reconstruites
        fig = plt.figure(figsize=(10, 4))
        fig.suptitle('Original (haut) vs. Reconstruit (bas)', fontsize=16)
        grid = ImageGrid(fig, 111, nrows_ncols=(2, 10), axes_pad=0.1)
        
        for i, ax in enumerate(grid):
            if i < 10:
                img = data[i].cpu().numpy().reshape(28, 28)
            else:
                img = recon[i-10].cpu().numpy().reshape(28, 28)
            ax.imshow(img, cmap='gray')
            ax.axis('off')
        plt.show()

# Afficher les reconstructions
visualize_reconstructions(model, test_loader, device)

In [7]:
def visualize_generated_samples(model, latent_dim, device, n_samples=25):
    model.eval()
    with torch.no_grad():
        # Echantillonner des points aleatoires de la distribution a priori (gaussienne standard)
        sample = torch.randn(n_samples, latent_dim).to(device)
        generated = model.decode(sample).cpu()
        
        fig, axes = plt.subplots(5, 5, figsize=(8, 8))
        fig.suptitle('Echantillons Generes a partir de l Espace Latent', fontsize=16)
        for i, ax in enumerate(axes.flat):
            ax.imshow(generated[i].view(28, 28), cmap='gray')
            ax.axis('off')
        plt.show()

# Afficher les echantillons generes
visualize_generated_samples(model, latent_dim, device)

Notebook executed (marker) — 2026-02-16 00:44:24


In [ ]:
def visualize_latent_space(model, data_loader, device):
    model.eval()
    latent_vectors = []
    labels = []
    with torch.no_grad():
        for data, label in data_loader:
            data = data.to(device)
            mu, _ = model.encode(data.view(-1, 784))
            latent_vectors.append(mu.cpu())
            labels.append(label.cpu())
            
    latent_vectors = torch.cat(latent_vectors).numpy()
    labels = torch.cat(labels).numpy()
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(latent_vectors[:, 0], latent_vectors[:, 1], c=labels, cmap='tab10')
    plt.colorbar(scatter, ticks=range(10))
    plt.xlabel('Dimension Latente 1')
    plt.ylabel('Dimension Latente 2')
    plt.title('Visualisation 2D de l Espace Latent')
    plt.show()

# Visualiser l'espace latent pour l'ensemble de test
visualize_latent_space(model, test_loader, device)